# Predictive Models

### Regression Analysis 
- **Predictive Academic Impact** - Modeling how sleep and social media hours predict academic performance.

In [ ]:
import polars as pl
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler

### Loading our cleaned data

In [136]:
from load_cleaned_data import load_cleaned_data

df = load_cleaned_data()

In [149]:
from sklearn.preprocessing import LabelEncoder

le = LabelEncoder()
df = df.with_columns(
    pl.Series('platform_usage', le.fit_transform(df['platform_usage'].to_numpy())),
    pl.Series('gender', le.fit_transform(df['gender'].to_numpy())),
    pl.Series('social_interaction_level', le.fit_transform(df['social_interaction_level'].to_numpy())),
)



In [144]:
# Preparing the data
# Select features and target
features = ['sleep_hours', 'daily_social_media_hours','screen_time_before_sleep','addiction_level','stress_level','anxiety_level','age','gender','platform_usage','social_interaction_level']
target = 'academic_performance'

# Convert to numpy (sklearn doesn't work with Polars directly)
X = df.select(features).to_numpy()
y = df.select(target).to_numpy().ravel()

In [145]:
# Split and Scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [146]:
# Train the model
model = LinearRegression()
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

In [147]:
# Evaluate
r2  = r2_score(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)

print(f"R² Score : {r2:.4f}")
print(f"MSE      : {mse:.4f}")
print(f"RMSE     : {rmse:.4f}")
print()
for feat, coef in zip(features, model.coef_):
    print(f"  {feat:30s} → coefficient: {coef:.4f}")
print(f"  {'Intercept':30s} → {model.intercept_:.4f}")

R² Score : -0.0144
MSE      : 0.3191
RMSE     : 0.5649

  sleep_hours                    → coefficient: 0.0243
  daily_social_media_hours       → coefficient: 0.0045
  screen_time_before_sleep       → coefficient: -0.0167
  addiction_level                → coefficient: 0.0316
  stress_level                   → coefficient: -0.0156
  anxiety_level                  → coefficient: -0.0360
  age                            → coefficient: 0.0092
  gender                         → coefficient: -0.0063
  platform_usage                 → coefficient: 0.0092
  social_interaction_level       → coefficient: 0.0033
  Intercept                      → 2.9809


In [148]:
# Visualize — Actual vs Predicted
fig = px.scatter(
    x=y_test,
    y=y_pred,
    labels={'x': 'Actual Academic Performance', 'y': 'Predicted Academic Performance'},
    title='Actual vs Predicted Academic Performance'
)

# Perfect prediction line
fig.add_shape(
    type='line',
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color='red', dash='dash')
)

fig.show()
fig.write_html("../visualizations/Actual vs Predicted Academic Performance.html ")

In [131]:
# Visualize — Feature Coefficients
coef_df = pl.DataFrame({
    'feature': features,
    'coefficient': model.coef_.tolist()
})

fig2 = px.bar(
    coef_df,
    x='feature',
    y='coefficient',
    color='coefficient',
    color_continuous_scale='RdYlGn',
    title='Feature Coefficients — Impact on Academic Performance'
)
fig2.show()
fig2.write_html("../visualizations/Feature Coefficients — Impact on Academic Performance.html ")

In [83]:
# Visualize — Residuals
residuals = y_test - y_pred

fig3 = px.scatter(
    x=y_pred,
    y=residuals,
    labels={'x': 'Predicted Values', 'y': 'Residuals'},
    title='Residual Plot'
)

# Zero residual line
fig3.add_hline(y=0, line_dash='dash', line_color='red')
fig3.show()
fig3.write_html("../visualizations/Residual Plot.html ")

In [179]:
# Metrics as a table
metrics_df = pl.DataFrame([lr_metrics])

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['Model', 'R²', 'MSE', 'RMSE'],
        fill_color='#2c3e50',
        font=dict(color='white', size=13),
        align='center',
        height=40
    ),
    cells=dict(
        values=[metrics_df[col] for col in metrics_df.columns],
        fill_color=[['#3498db', '#e67e22']],  # blue = LR, orange = RF
        font=dict(color='white', size=12),
        align='center',
        height=35
    )
)])

fig_table.update_layout(title='Model Metrics — Linear Regression ')
fig_table.show()
fig_table.write_html("../visualizations/Model Metrics — Linear Regression .html ")

## Random Forest

In [71]:
# Train Random Forest
from sklearn.ensemble import RandomForestRegressor

rf_model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1  # use all CPU cores
)

rf_model.fit(X_train_scaled, y_train)
rf_pred = rf_model.predict(X_test_scaled)

In [72]:
#Evaluate both models
def evaluate_model(name, y_test, y_pred):
    r2   = r2_score(y_test, y_pred)
    mse  = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    print(f"--- {name} ---")
    print(f"  R²   : {r2:.4f}")
    print(f"  MSE  : {mse:.4f}")
    print(f"  RMSE : {rmse:.4f}\n")
    return {'Model': name, 'R²': round(r2, 4), 'MSE': round(mse, 4), 'RMSE': round(rmse, 4)}

lr_metrics = evaluate_model('Linear Regression', y_test, y_pred)
rf_metrics = evaluate_model('Random Forest',     y_test, rf_pred)

--- Linear Regression ---
  R²   : -0.0173
  MSE  : 0.3200
  RMSE : 0.5657

--- Random Forest ---
  R²   : -0.0738
  MSE  : 0.3378
  RMSE : 0.5812



In [84]:
# Compare metrics as a table
metrics_df = pl.DataFrame([lr_metrics, rf_metrics])

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['Model', 'R²', 'MSE', 'RMSE'],
        fill_color='#2c3e50',
        font=dict(color='white', size=13),
        align='center',
        height=40
    ),
    cells=dict(
        values=[metrics_df[col] for col in metrics_df.columns],
        fill_color=[['#3498db', '#e67e22']],  # blue = LR, orange = RF
        font=dict(color='white', size=12),
        align='center',
        height=35
    )
)])

fig_table.update_layout(title='Model Comparison — Linear Regression vs Random Forest')
fig_table.show()
fig_table.write_html("../visualizations/Model Comparison — Linear Regression vs Random Forest.html ")


In [85]:
# Actual vs Predicted - For  both Models
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=y_test, y=y_pred,
    mode='markers',
    name='Linear Regression',
    marker=dict(color='#3498db', opacity=0.6)
))

fig.add_trace(go.Scatter(
    x=y_test, y=rf_pred,
    mode='markers',
    name='Random Forest',
    marker=dict(color='#e67e22', opacity=0.6)
))

# Perfect prediction line
fig.add_shape(
    type='line',
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color='red', dash='dash')
)

fig.update_layout(
    title='Actual vs Predicted — Linear Regression vs Random Forest',
    xaxis_title='Actual Academic Performance',
    yaxis_title='Predicted Academic Performance'
)
fig.show()
fig.write_html("../visualizations/Actual vs Predicted — Linear Regression vs Random Forest.html ")

In [86]:
# Feature Importance - Random Forest Only
importance_df = pl.DataFrame({
    'feature'   : features,
    'importance': rf_model.feature_importances_.tolist()
}).sort('importance', descending=True)

fig2 = px.bar(
    importance_df,
    x='feature',
    y='importance',
    color='importance',
    color_continuous_scale='Oranges',
    title='Random Forest — Feature Importance'
)
fig2.show()
fig2.write_html("../visualizations/Random Forest — Feature Importance.html ")

In [87]:
# Residuals - Both Models
fig3 = go.Figure()

fig3.add_trace(go.Scatter(
    x=y_pred, y=y_test - y_pred,
    mode='markers',
    name='Linear Regression',
    marker=dict(color='#3498db', opacity=0.6)
))

fig3.add_trace(go.Scatter(
    x=rf_pred, y=y_test - rf_pred,
    mode='markers',
    name='Random Forest',
    marker=dict(color='#e67e22', opacity=0.6)
))

fig3.add_hline(y=0, line_dash='dash', line_color='red')

fig3.update_layout(
    title='Residuals — Linear Regression vs Random Forest',
    xaxis_title='Predicted Values',
    yaxis_title='Residuals'
)
fig3.show()
fig3.write_html("../visualizations/Residuals — Linear Regression vs Random Forest.html ")

## XGBOOST

In [151]:
from xgboost import XGBRegressor

xgb_model = XGBRegressor(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=4,
    random_state=42,
    n_jobs=-1
)

xgb_model.fit(X_train_scaled, y_train)
xgb_pred = xgb_model.predict(X_test_scaled)

In [152]:
lr_metrics  = evaluate_model('Linear Regression', y_test, y_pred)
rf_metrics  = evaluate_model('Random Forest',     y_test, rf_pred)
xgb_metrics = evaluate_model('XGBoost',           y_test, xgb_pred)

--- Linear Regression ---
  R²   : -0.0144
  MSE  : 0.3191
  RMSE : 0.5649

--- Random Forest ---
  R²   : -0.0738
  MSE  : 0.3378
  RMSE : 0.5812

--- XGBoost ---
  R²   : -0.1337
  MSE  : 0.3566
  RMSE : 0.5972



In [183]:
metrics_df = pl.DataFrame([lr_metrics, rf_metrics, xgb_metrics])

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=['Model', 'R²', 'MSE', 'RMSE'],
        fill_color='#2c3e50',
        font=dict(color='white', size=13),
        align='center',
        height=40
    ),
    cells=dict(
        values=[metrics_df[col] for col in metrics_df.columns],
        fill_color=[['#3498db', '#e67e22', '#9b59b6']],  # blue, orange, purple
        font=dict(color='white', size=12),
        align='center',
        height=35
    )
)])

fig_table.update_layout(title='Model Comparison — LR vs RF vs XGBoost')
fig_table.show()
fig_table.write_html("../visualizations/XGBoost - Model Comparison — LR vs RF vs XGBoost.html ")

In [182]:
fig = go.Figure()

for name, preds, color in [
    ('Linear Regression', y_pred,   '#3498db'),
    ('Random Forest',     rf_pred,  '#e67e22'),
    ('XGBoost',           xgb_pred, '#9b59b6')
]:
    fig.add_trace(go.Scatter(
        x=y_test, y=preds,
        mode='markers',
        name=name,
        marker=dict(color=color, opacity=0.6)
    ))

fig.add_shape(
    type='line',
    x0=y_test.min(), y0=y_test.min(),
    x1=y_test.max(), y1=y_test.max(),
    line=dict(color='red', dash='dash')
)

fig.update_layout(
    title='Actual vs Predicted — All Models',
    xaxis_title='Actual Academic Performance',
    yaxis_title='Predicted Academic Performance'
)
fig.show()
fig.write_html("../visualizations/XGBoost - Actual vs Predicted — All Models.html ")

In [155]:
importance_df = pl.DataFrame({
    'feature'   : features,
    'importance': xgb_model.feature_importances_.tolist()
}).sort('importance', descending=True)

fig2 = px.bar(
    importance_df,
    x='feature',
    y='importance',
    color='importance',
    color_continuous_scale='Purples',
    title='XGBoost — Feature Importance'
)
fig2.show()
fig2.write_html("../visualizations/XGBoost - Feature Importance.html ")

In [162]:
df.schema

Schema([('age', Int64),
        ('gender', Int64),
        ('daily_social_media_hours', Float64),
        ('platform_usage', Int64),
        ('sleep_hours', Float64),
        ('screen_time_before_sleep', Float64),
        ('academic_performance', Float64),
        ('physical_activity', Float64),
        ('social_interaction_level', Int64),
        ('stress_level', Int64),
        ('anxiety_level', Int64),
        ('addiction_level', Int64),
        ('depression_label', Int64)])

In [166]:
print(df.to_pandas().corr()["anxiety_level"].sort_values())

academic_performance       -0.063861
physical_activity          -0.022499
gender                     -0.013939
stress_level               -0.013311
screen_time_before_sleep   -0.008377
social_interaction_level   -0.004874
daily_social_media_hours   -0.002154
sleep_hours                 0.020870
age                         0.025421
platform_usage              0.032143
addiction_level             0.038732
anxiety_level               1.000000
depression_label                 NaN
Name: anxiety_level, dtype: float64


# K-Means Clustering pipeline for your teen social media dataset

In [167]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
import plotly.express as px
import plotly.graph_objects as go
import polars as pl
import numpy as np# 

In [168]:
# Select features and scale
features = [
    'daily_social_media_hours',
    'addiction_level',
    'sleep_hours',
    'stress_level',
    'academic_performance'
]

X = df.select(features).to_numpy()

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [180]:
# Find optimal K— Elbow + Silhouette
inertias    = []
silhouettes = []
k_range     = range(2, 11)

for k in k_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_scaled)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_scaled, km.labels_))

# Elbow plot
fig_elbow = go.Figure()
fig_elbow.add_trace(go.Scatter(
    x=list(k_range), y=inertias,
    mode='lines+markers',
    line=dict(color='#3498db'),
    name='Inertia'
))
fig_elbow.update_layout(
    title='Elbow Method — Optimal K',
    xaxis_title='Number of Clusters (K)',
    yaxis_title='Inertia'
)
fig_elbow.show()
fig_elbow.write_html("../visualizations/K-Means Clustering - Find optimal K— Elbow.html ")

# Silhouette plot
fig_sil = go.Figure()
fig_sil.add_trace(go.Scatter(
    x=list(k_range), y=silhouettes,
    mode='lines+markers',
    line=dict(color='#e67e22'),
    name='Silhouette Score'
))
fig_sil.update_layout(
    title='Silhouette Score — Optimal K',
    xaxis_title='Number of Clusters (K)',
    yaxis_title='Silhouette Score (higher = better)'
)
fig_sil.show()
fig_sil.write_html("../visualizations/K-Means Clustering - Find optimal K—  Silhouette.html ")

In [170]:
# Train Final Model
# Set K based on elbow/silhouette results (adjust as needed)
OPTIMAL_K = 4

km_final = KMeans(n_clusters=OPTIMAL_K, random_state=42, n_init=10)
km_final.fit(X_scaled)

# Add cluster labels back to dataframe
df = df.with_columns(
    pl.Series('cluster', km_final.labels_.astype(str))
)

In [ ]:
# Visualize clusters - Scatter
fig_scatter = px.scatter(
    df.to_pandas(),
    x='daily_social_media_hours',
    y='academic_performance',
    color='cluster',
    symbol='cluster',
    size='addiction_level',
    hover_data=['stress_level', 'sleep_hours'],
    color_discrete_sequence=['#3498db', '#e67e22', '#9b59b6', '#2ecc71'],
    title='K-Means Clusters — Screen Time vs Academic Performance'
)
fig_scatter.show()
fig_scatter.write_html("../visualizations/K-Means Clustering -- Visualize Clusters.html ")

In [181]:
# Cluster Profiles - Radar Chart
cluster_profiles = df.group_by('cluster').agg([
    pl.col('daily_social_media_hours').mean(),
    pl.col('addiction_level').mean(),
    pl.col('sleep_hours').mean(),
    pl.col('stress_level').mean(),
    pl.col('academic_performance').mean()
]).sort('cluster')

# Melt for radar
cluster_melted = cluster_profiles.unpivot(
    index='cluster',
    on=features,
    variable_name='metric',
    value_name='value'
)

fig_radar = px.line_polar(
    cluster_melted.to_pandas(),
    r='value',
    theta='metric',
    color='cluster',
    line_close=True,
    color_discrete_sequence=['#3498db', '#e67e22', '#9b59b6', '#2ecc71'],
    title='Cluster Profiles — Radar Chart'
)
fig_radar.show()
fig_radar.write_html("../visualizations/K-Means Clustering - Cluster Profiles - Radar Chart.html ")

In [186]:
# Cluster summary table
cluster_summary = df.group_by('cluster').agg([
    pl.len().alias('user_count'),
    pl.col('daily_social_media_hours').mean().round(2).alias('avg_screen_time'),
    pl.col('addiction_level').mean().round(2).alias('avg_addiction'),
    pl.col('sleep_hours').mean().round(2).alias('avg_sleep'),
    pl.col('stress_level').mean().round(2).alias('avg_stress'),
    pl.col('academic_performance').mean().round(2).alias('avg_academic')
]).sort('cluster')

cluster_summary

cluster,user_count,avg_screen_time,avg_addiction,avg_sleep,avg_stress,avg_academic
str,u32,f64,f64,f64,f64,f64
"""0""",290,2.84,5.2,7.79,4.49,3.1
"""1""",308,6.08,4.77,6.04,2.94,2.95
"""2""",256,6.18,5.66,7.18,8.27,3.04
"""3""",315,3.05,6.63,5.19,6.18,2.88


In [188]:
# Compare cluster metrics as a styled summary table

import polars as pl
import plotly.graph_objects as go

# Assuming cluster_summary already exists
metrics_df = pl.DataFrame(cluster_summary)

fig_table = go.Figure(data=[go.Table(
    header=dict(
        values=[
            'Cluster',
            'User Count',
            'AVG Screen Time',
            'AVG Addiction',
            'AVG Sleep',
            'AVG Stress',
            'AVG Academic Performance'
        ],
        fill_color='#2c3e50',
        font=dict(color='white', size=13),
        align='center',
        height=40
    ),
    
    cells=dict(
        values=[metrics_df[col].to_list() for col in metrics_df.columns],
        fill_color=[
            ['#3498db', '#2980b9', '#3498db', '#2980b9']  # alternating row colors
        ],
        font=dict(color='white', size=12),
        align='center',
        height=35
    )
)])

fig_table.update_layout(
    title='K-Means Clustering — Cluster Summary Table',
    title_x=0.5,
    width=1000,
    height=400
)

fig_table.show()

# Optional save
fig_table.write_html("../visualizations/kmeans_cluster_summary_table.html")

In [ ]:
print(df.to_pandas().corr()["anxiety_level"].sort_values())

academic_performance       -0.063861
physical_activity          -0.022499
gender                     -0.013939
stress_level               -0.013311
screen_time_before_sleep   -0.008377
social_interaction_level   -0.004874
daily_social_media_hours   -0.002154
sleep_hours                 0.020870
age                         0.025421
platform_usage              0.032143
addiction_level             0.038732
anxiety_level               1.000000
depression_label                 NaN
Name: anxiety_level, dtype: float64
